# Experiment: Concurrent agents middleware

Análise local da curva closed-loop do runtime nu (G2) e da fatia pelo gateway (G4),
com o joelho operacional já aceito (D17).

- Entradas: `services/inference-runtime/results-from-pod/g2/` e `.../g4/`
- Spec: `specs/concurrent-agents-middleware/spec.md`
- Sem efeito remoto: só lê artefatos já puxados do Pod.
- Não mede de novo; não altera resultados brutos.


## 0. Configuração

Aponte `RESULTS_ROOT` para a cópia local. O notebook falha cedo se faltar célula
esperada (81 no G2 cache-on; 15 no G4 via gateway).


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

REPO = Path("..").resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Prefer repo root discovery from this notebook path.
HERE = Path.cwd()
for candidate in [HERE, *HERE.parents]:
    if (candidate / "specs" / "concurrent-agents-middleware" / "spec.md").is_file():
        REPO = candidate
        break

RESULTS_ROOT = REPO / "services" / "inference-runtime" / "results-from-pod"
# Prefer the pulled trees; fall back to the local archive folder used after G4.
if (RESULTS_ROOT / "g2").is_dir() and (RESULTS_ROOT / "g4").is_dir():
    G2_ROOT = RESULTS_ROOT / "g2"
    G4_ROOT = RESULTS_ROOT / "g4"
else:
    G2_ROOT = RESULTS_ROOT / "ge_g4" / "g2"
    G4_ROOT = RESULTS_ROOT / "ge_g4" / "g4"

EXPECTED_G2 = 81  # 3 planos × 9 streams × 3 reps
EXPECTED_G4 = 15  # 1 plano × 5 streams × 3 reps
STREAMS_ORDER = [1, 2, 4, 6, 8, 12, 16, 24, 32]
KNEE_PARALLEL = 8
KNEE_STREAMS_MAX = 8

display(Markdown(f"**REPO** `{REPO}`  \n**G2** `{G2_ROOT}`  \n**G4** `{G4_ROOT}`"))


## 1. Carregar manifests

Cada célula G2/G4 grava `manifest.json` com `summary` (req/s, tok/s, TTFT, E2E).
Pastas cache-off ficam de fora se aparecerem.


In [ ]:
CELL_RE = re.compile(
    r"p(?P<parallel>\d+)-ctx(?P<ctx>\d+)-cache-(?P<cache>on|off)(?P<via>-via-gateway)?/s(?P<streams>\d+)-r(?P<rep>\d+)$"
)


def load_cells(root: Path) -> pd.DataFrame:
    rows = []
    for manifest_path in sorted(root.rglob("manifest.json")):
        rel = manifest_path.parent.relative_to(root).as_posix()
        match = CELL_RE.search(rel)
        if match is None:
            continue
        payload = json.loads(manifest_path.read_text())
        summary = payload.get("summary") or {}
        rows.append(
            {
                "path": rel,
                "parallel": int(payload.get("parallel", match.group("parallel"))),
                "ctx_size": int(payload.get("ctx_size", match.group("ctx"))),
                "cache": payload.get("cache", match.group("cache")),
                "via": payload.get("via", "gateway" if match.group("via") else "runtime"),
                "streams": int(payload.get("streams", match.group("streams"))),
                "repetition": int(payload.get("repetition", match.group("rep"))),
                "ok": int(summary.get("successful_count", 0)),
                "err": int(summary.get("errored_count", 0)),
                "req_s": float(summary.get("requests_per_second", float("nan"))),
                "tok_s": float(summary.get("output_tokens_per_second", float("nan"))),
                "ttft_p50": float(summary.get("ttft_ms_p50", float("nan"))),
                "ttft_p95": float(summary.get("ttft_ms_p95", float("nan"))),
                "e2e_p50": float(summary.get("e2e_s_p50", float("nan"))),
                "e2e_p95": float(summary.get("e2e_s_p95", float("nan"))),
                "prompt_tokens_missing": int(summary.get("prompt_tokens_missing", 0)),
                "health_before": payload.get("health_before"),
                "health_after": payload.get("health_after"),
                "gguf_sha256": (payload.get("gguf_sha256") or "")[:12],
                "client_sha256": (payload.get("client_sha256") or "")[:8],
            }
        )
    return pd.DataFrame(rows)


g2 = load_cells(G2_ROOT)
g4 = load_cells(G4_ROOT)

assert len(g2) == EXPECTED_G2, f"G2 esperado {EXPECTED_G2}, achou {len(g2)}"
assert len(g4) == EXPECTED_G4, f"G4 esperado {EXPECTED_G4}, achou {len(g4)}"
assert (g2["ok"] == 128).all() and (g2["err"] == 0).all()
assert (g4["ok"] == 128).all() and (g4["err"] == 0).all()
assert (g2["via"] == "runtime").all()
assert (g4["via"] == "gateway").all()
assert set(g2["parallel"]) == {1, 8, 16}
assert set(g4["parallel"]) == {8}

display(Markdown(
    f"G2: **{len(g2)}** células · G4: **{len(g4)}** células · "
    f"erros: **{int(g2['err'].sum() + g4['err'].sum())}**"
))
g2.head(3)


## 2. Curva G2 (runtime nu) e joelho

Médias das três repetições. O joelho operacional (D17) é `--parallel 8` com
streams ≤ 8: a taxa sobe até ali e o TTFT p50 fica abaixo de ~400 ms.


In [ ]:
def mean_curve(frame: pd.DataFrame) -> pd.DataFrame:
    cols = ["req_s", "tok_s", "ttft_p50", "ttft_p95", "e2e_p50", "e2e_p95"]
    out = (
        frame.groupby(["parallel", "streams"], as_index=False)[cols]
        .mean()
        .sort_values(["parallel", "streams"])
    )
    return out


g2_curve = mean_curve(g2)
g2_p8 = g2_curve[g2_curve["parallel"] == KNEE_PARALLEL].copy()

display(Markdown("### Médias G2 por plano"))
for parallel, block in g2_curve.groupby("parallel"):
    display(Markdown(f"**parallel={parallel}**"))
    show = block.copy()
    show["req_s"] = show["req_s"].round(3)
    show["tok_s"] = show["tok_s"].round(1)
    show["ttft_p50"] = show["ttft_p50"].round(1)
    show["e2e_p50"] = show["e2e_p50"].round(3)
    display(show[["streams", "req_s", "tok_s", "ttft_p50", "e2e_p50"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for parallel, block in g2_curve.groupby("parallel"):
    axes[0].plot(block["streams"], block["req_s"], marker="o", label=f"p={parallel}")
    axes[1].plot(block["streams"], block["ttft_p50"], marker="o", label=f"p={parallel}")
axes[0].axvline(KNEE_STREAMS_MAX, color="0.5", ls="--", lw=1)
axes[1].axvline(KNEE_STREAMS_MAX, color="0.5", ls="--", lw=1)
axes[0].set_xlabel("streams"); axes[0].set_ylabel("req/s"); axes[0].set_title("G2 taxa")
axes[1].set_xlabel("streams"); axes[1].set_ylabel("TTFT p50 (ms)"); axes[1].set_title("G2 TTFT")
axes[0].legend(); axes[1].legend()
fig.tight_layout()
plt.show()

peak_p8 = float(g2_p8["req_s"].max())
at8 = g2_p8[g2_p8["streams"] == 8].iloc[0]
display(Markdown(
    f"Joelho **p8 / streams≤8**: req/s@8 ≈ **{at8['req_s']:.2f}**, "
    f"TTFT p50 ≈ **{at8['ttft_p50']:.0f} ms**, pico do plano ≈ **{peak_p8:.2f}** req/s "
    f"(em streams={int(g2_p8.loc[g2_p8['req_s'].idxmax(), 'streams'])})."
))


## 3. G4 vs G2 no plano de 8 slots

Mesma fatia de streams 1–8. G4 passa pelo gateway (`via=gateway`). Pods
diferentes: a leitura é forma e ordem de grandeza, não igualdade bit a bit.


In [ ]:
g4_curve = mean_curve(g4)
g2_knee = g2_p8[g2_p8["streams"].isin(sorted(g4_curve["streams"]))].copy()

cmp = g2_knee.merge(
    g4_curve,
    on="streams",
    suffixes=("_g2", "_g4"),
)
cmp["req_ratio"] = cmp["req_s_g4"] / cmp["req_s_g2"]
cmp["ttft_delta_ms"] = cmp["ttft_p50_g4"] - cmp["ttft_p50_g2"]
cmp["e2e_delta_s"] = cmp["e2e_p50_g4"] - cmp["e2e_p50_g2"]

show = cmp[
    ["streams", "req_s_g4", "req_s_g2", "req_ratio", "ttft_p50_g4", "ttft_p50_g2", "ttft_delta_ms", "e2e_p50_g4", "e2e_p50_g2"]
].copy()
for col in ["req_s_g4", "req_s_g2", "req_ratio", "e2e_p50_g4", "e2e_p50_g2"]:
    show[col] = show[col].round(3)
for col in ["ttft_p50_g4", "ttft_p50_g2", "ttft_delta_ms"]:
    show[col] = show[col].round(1)
display(show)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(cmp["streams"], cmp["req_s_g2"], marker="o", label="G2 nu")
axes[0].plot(cmp["streams"], cmp["req_s_g4"], marker="s", label="G4 gateway")
axes[1].plot(cmp["streams"], cmp["ttft_p50_g2"], marker="o", label="G2 nu")
axes[1].plot(cmp["streams"], cmp["ttft_p50_g4"], marker="s", label="G4 gateway")
axes[0].set_xlabel("streams"); axes[0].set_ylabel("req/s"); axes[0].set_title("Taxa no joelho")
axes[1].set_xlabel("streams"); axes[1].set_ylabel("TTFT p50 (ms)"); axes[1].set_title("TTFT no joelho")
axes[0].legend(); axes[1].legend()
fig.tight_layout()
plt.show()

display(Markdown(
    f"Overhead típico do gateway: taxa **{cmp['req_ratio'].min():.2f}–{cmp['req_ratio'].max():.2f}×** do runtime nu; "
    f"TTFT **+{cmp['ttft_delta_ms'].min():.0f}–{cmp['ttft_delta_ms'].max():.0f} ms**."
))


## 4. Conteúdo sob concorrência (G2c parcial)

Leitura já registrada em `acceptance.md`: repetições sequenciais batem byte a
byte; com parallel > 1 o batch muda tags em ~12–15% dos casos. Isso não prova
troca de respostas (`request_id` permanece alinhado). Eco deliberado e Macro-F1
contra labels ainda estão abertos e **não** entram neste notebook até o CSV de
calibração e o cliente de eco existirem na sessão.


## 5. Conclusão e `hypothesis_outcome`

Hipótese: middleware com admissão/fila/justiça mantém o runtime disponível e as
respostas íntegras sob carga, com degradação previsível.

Evidência deste corte:

1. Runtime nu não travou nem corrompeu health nas 81 células cache-on.
2. Degradação é previsível: com 1 slot a taxa satura cedo; com 8 slots sobe até
   ~4 req/s e a fila aparece acima de 8 streams.
3. 16 slots quase não aumentam taxa (~3% no pico); justificam-se só para TTFT
   com streams > 8 — fora do ponto operacional.
4. Gateway no joelho preserva a forma da curva com custo pequeno (~3–8% taxa,
   +50–100 ms TTFT) e rejeições explícitas cobertas por testes locais.
5. Justificativa principal do middleware neste corte: **admissão, justiça,
   backpressure e observabilidade** — não a prevenção de hang do llama-server
   sozinho no regime medido.

`hypothesis_outcome` proposto para a spec: **supported-with-nuance** —
a hipótese nula (runtime sozinho já sustenta N sem falha observável de
disponibilidade) **não foi refutada** no joelho; o middleware ainda se justifica
pelos controles explícitos acima.
